# 动态 beta 上限研究

冻结策略把有效 beta 硬封在常数 0.6。本卡测试用**随已实现波动缩放的动态上限**替代常数，
只重投影冻结信号，不改模型。development 截至 2022-12-31，其后为 outer。


In [1]:
import sys, hashlib, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT/'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
CARD = ROOT/'research'/'stage1_dynamic_beta_cap_20261010'
plan_bytes=(CARD/'plan.json').read_bytes(); plan=json.loads(plan_bytes)
print('plan sha256:', hashlib.sha256(plan_bytes).hexdigest())
print('source card:', plan['source_card'])


plan sha256: bafb76d7cfe3ead24e7c4d1b5bc0326fac100a6db1efc3cda216addd8b0b4b7e
source card: pit_constituent_auxiliary_training_allocation_retrained_20260928


In [2]:
from src.workflow import dynamic_beta_cap as workflow
from src.workflow.dynamic_beta_cap import write_card_files
summary = workflow.run_dynamic_beta_cap(root=ROOT)
write_card_files(summary, card_dir=CARD)
print('decision:', summary['decision'], '| selected:', summary['selected_rule'], '| improvement:', summary['improvement'])
print('recorder:', summary['recorder_id'])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


2026/10/09 20:02:30 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[54788:MainThread](2026-10-09 20:02:30,359) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[54788:MainThread](2026-10-09 20:02:30,361) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[54788:MainThread](2026-10-09 20:02:30,362) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[54788:MainThread](2026-10-09 20:02:30,369) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_dynamic_beta_cap_20261010.


[54788:MainThread](2026-10-09 20:02:30,371) INFO - qlib.workflow - [exp.py:258] - Experiment 634190272876448090 starts running ...


[54788:MainThread](2026-10-09 20:02:30,469) INFO - qlib.workflow - [recorder.py:345] - Recorder af44025ef57a4eeaad56806e2e9500a8 starts running under Experiment 634190272876448090 ...


[54788:MainThread](2026-10-09 20:02:30,565) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


decision: dynamic_beta_cap_improves_out_of_sample | selected: vol_w60_s6 | improvement: True
recorder: af44025ef57a4eeaad56806e2e9500a8


In [3]:
import pandas as pd
t = pd.read_csv(CARD/'beta_cap_comparison.csv')
display(t[t.cost_bps==10.0][['rule','avg_cap','sharpe','dev_sharpe','outer_sharpe','return_pct','max_drawdown_pct','annual_win_count','annual_years']].round(3))


,rule,avg_cap,sharpe,dev_sharpe,outer_sharpe,return_pct,max_drawdown_pct,annual_win_count,annual_years
0,constant_0.6,0.600,0.928,0.737,1.236,450.713,-31.679,NaN,11
1,vol_w30_s6,0.638,1.041,0.880,1.332,424.977,-25.102,6.0,11
2,vol_w30_s10,1.023,0.992,0.926,1.108,892.870,-38.676,5.0,11
3,vol_w60_s6,0.621,1.013,0.882,1.253,416.551,-25.698,4.0,11
4,vol_w60_s10,1.013,0.935,0.868,1.053,815.390,-39.523,5.0,11
5,vol_w120_s6,0.635,0.929,0.793,1.180,402.802,-28.366,3.0,11
6,vol_w120_s10,1.046,0.881,0.784,1.054,820.689,-42.903,4.0,11


## 中文结论


In [4]:
print('选中规则:', summary['selected_rule'])
print('outer 是否改善:', summary['improvement'])
print('决定:', summary['decision'])


选中规则: vol_w60_s6
outer 是否改善: True
决定: dynamic_beta_cap_improves_out_of_sample
